In [14]:
import os
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, TimeSeriesSplit, RandomizedSearchCV
from tabpfn import TabPFNRegressor
from tensorflow.python.ops.metrics_impl import root_mean_squared_error
from xgboost import XGBRegressor

def train_and_evaluate_random_search(model, param_grid, X_train, y_train, X_test, y_test, cv_splitter, model_name):
    print(f"\n--- {model_name} Training ---")

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        cv=cv_splitter,
        n_iter=50,
        n_jobs=-1,
        scoring="neg_mean_squared_error"
    )

    search.fit(X_train, y_train)

    print(f"Tuning finished. Best Params are the following:")
    for param, value in search.best_params_.items():
        print(f"  {param}: {value}")

    best_model = search.best_estimator_
    preds = best_model.predict(X_test)

    print(f"Results Test:")
    print(f"MAE: {mean_absolute_error(y_test, preds)}")
    print(f"RMSE: {root_mean_squared_error(y_test, preds)}")
    print(f"R2:  {r2_score(y_test, preds)}")

    return best_model, preds

def train_tabpfn(X_train, y_train, X_test, y_test):
    print("\n--- TabPFN Training ---")
    os.environ["TABPFN_TOKEN"] = "tabpfn_sk_NUj6JOOu4z6GJkEQJSM-Z67PWJPjSTwXU7pFZCZgzVg"

    model = TabPFNRegressor()
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    print(f"Results Test:")
    print(f"MAE: {mean_absolute_error(y_test, preds):.4f}")
    print(f"RMSE: {root_mean_squared_error(y_test, preds)}")
    print(f"R2:  {r2_score(y_test, preds):.4f}")

    return model, preds

if __name__ == "__main__":

    df = pd.read_csv("./data/final_dataset.csv")
    df["planningDate_dt"] = pd.to_datetime(df["planningDate_dt"])
    df = df.sort_values(by="planningDate_dt").reset_index(drop=True)

    y = df["totalAssignments"]
    X_matrix = df.drop(columns=["totalAssignments", "planningDate_dt"], axis=1)

    X_train, X_test, y_train, y_test = train_test_split(X_matrix, y, test_size=0.3, random_state=42, shuffle=False)
    time_split = TimeSeriesSplit(n_splits=3)

    rf_params = {
        "n_estimators": [50, 100, 150, 200, 250, 300],
        "max_depth": [5, 8, 12, 15],
        "min_samples_split": [2, 4, 6, 8],
        "max_features": ["sqrt", "log2", 0.5],
    }
    rf_model, rf_preds = train_and_evaluate_random_search(
        RandomForestRegressor(random_state=42, n_jobs=-1),
        rf_params, X_train, y_train, X_test, y_test, time_split, "Random Forest"
    )

    xgb_params = {
        "n_estimators": [50, 100, 150, 200],
        "learning_rate": [0.01, 0.05, 0.1],
        "max_depth": [3, 5, 7, 9],
        "subsample": [0.4, 0.6, 0.8, 1.0],
        "colsample_bytree": [0.4, 0.6, 0.8, 1.0],
        "reg_lambda": [1, 5, 10, 15],
    }
    xgb_model, xgb_preds = train_and_evaluate_random_search(
        XGBRegressor(random_state=42, n_jobs=-1, objective="reg:squarederror"),
        xgb_params, X_train, y_train, X_test, y_test, time_split, "XGBoost"
    )

    tabpfn_model, tabpfn_preds = train_tabpfn(X_train, y_train, X_test, y_test)

-------------------------
1- Random Forest Training
Random Forest Tuning completed. Best params are the following:
n_estimators: 50
min_samples_split: 8
max_features: 0.5
max_depth: 15
Random Forest Test completed. Results are the following:
Mean Absolute Error: 5.940310375297217
R2: 0.8764884878250055
-------------------------
2- eXtream Gradient Boosting Training
XGBoost Tuning completed. Best params are the following:
subsample: 0.6
reg_lambda: 5
n_estimators: 200
max_depth: 3
learning_rate: 0.1
colsample_bytree: 1.0
XGBoost Test completed. Results are the following:
Mean Absolute Error: 6.186348915100098
R2: 0.8463398814201355
-------------------------
3- TabPFN Training
TabPFN Test completed. Results are the following:
Mean Absolute Error: 3.4315547943115234
R2: 0.9579998850822449
